In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# This ensures your plots show up directly inside the notebook
%matplotlib inline

In [15]:
!pip install --upgrade numexpr bottleneck

In [5]:
df = pd.read_csv('E:\Project datasets\Bengaluru_House_Data.csv')

# Display the first 5 rows
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [9]:
df.shape

(13320, 9)

In [7]:
# Display the count of missing values for each column
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [11]:
df2 = df.drop(['society', 'balcony', 'availability', 'area_type'], axis='columns')

In [13]:
df3 = df2.dropna()

In [15]:
print("New Shape:", df3.shape)
print("\nMissing values now:\n", df3.isnull().sum())

New Shape: (13246, 5)

Missing values now:
 location      0
size          0
total_sqft    0
bath          0
price         0
dtype: int64


In [17]:
# 1. Let's see all the unique and messy ways people entered the size
print("Unique values in 'size' column:")
print(df3['size'].unique())

# 2. Create a new column called 'bhk' by taking the string, splitting it by the space, and keeping the first number
df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))

# 3. Check the new column alongside the old one
df3[['size', 'bhk']].head()

Unique values in 'size' column:
<ArrowStringArray>
[     '2 BHK',  '4 Bedroom',      '3 BHK',      '4 BHK',  '6 Bedroom',
  '3 Bedroom',      '1 BHK',       '1 RK',  '1 Bedroom',  '8 Bedroom',
  '2 Bedroom',  '7 Bedroom',      '5 BHK',      '7 BHK',      '6 BHK',
  '5 Bedroom',     '11 BHK',      '9 BHK',  '9 Bedroom',     '27 BHK',
 '10 Bedroom', '11 Bedroom',     '10 BHK',     '19 BHK',     '16 BHK',
 '43 Bedroom',     '14 BHK',      '8 BHK', '12 Bedroom',     '13 BHK',
 '18 Bedroom']
Length: 31, dtype: str


,size,bhk
0,2 BHK,2
1,4 Bedroom,4
2,3 BHK,3
3,3 BHK,3
4,2 BHK,2


In [19]:
df3['bhk'].unique()

array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18], dtype=int64)

In [21]:
# quick helper function to check if a value is a normal decimal number
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

# 2. Look at all the rows where 'total_sqft' is NOT a normal number
# (The '~' symbol means "NOT")
df3[~df3['total_sqft'].apply(is_float)].head(10)

,location,size,total_sqft,bath,price,bhk
30,Yelahanka,4 BHK,2100 - 2850,4.0,186.000,4
122,Hebbal,4 BHK,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,2 BHK,1042 - 1105,2.0,54.005,2
165,Sarjapur,2 BHK,1145 - 1340,2.0,43.490,2
188,KR Puram,2 BHK,1015 - 1540,2.0,56.800,2
410,Kengeri,1 BHK,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,2 BHK,1195 - 1440,2.0,63.770,2
648,Arekere,9 Bedroom,4125Perch,9.0,265.000,9
661,Yelahanka,2 BHK,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,4 Bedroom,3090 - 5002,4.0,445.000,4


In [23]:
# 1. Define a function to convert the messy ranges into a single average number
def convert_sqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None   # We return None for the weird text ones to drop them later

# 2. Apply the function to create a new clean DataFrame
df4 = df3.copy()
df4['total_sqft'] = df4['total_sqft'].apply(convert_sqft_to_num)

# 3. Drop the few rows that returned 'None' and check the  work
df4 = df4.dropna()
print("Cleaned shape:", df4.shape)

Cleaned shape: (13200, 6)


In [25]:
# 1. Create a fresh copy of the dataframe
df5 = df4.copy()

# 2. Calculate the price per square foot
df5['price_per_sqft'] = df5['price'] * 100000 / df5['total_sqft']

# 3. Display the first few rows to verify the new column
df5.head()

,location,size,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2,4250.000000


In [27]:
# Strip any accidental hidden spaces from the location names
df5.location = df5.location.apply(lambda x: x.strip())

# Count exactly how many unique locations exist
location_stats = df5.groupby('location')['location'].agg('count').sort_values(ascending=False)
print("Total unique locations:", len(location_stats))

# Look at the top 10 most popular locations
location_stats.head(10)

Total unique locations: 1287


location
Whitefield               533
Sarjapur  Road           392
Electronic City          304
Kanakpura Road           264
Thanisandra              235
Yelahanka                210
Uttarahalli              186
Hebbal                   176
Marathahalli             175
Raja Rajeshwari Nagar    171
Name: location, dtype: int64